# PCG Convergence Study  
Using NGSolve's built-in pcg solver and our finite element space hierarchy  


In [1]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
import time
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.multigrid_cycles import (
    build_form_setup,
    build_hierarchy,
    MultigridSolver,
    VCycleConfig,
)

In [2]:
# Getting colors for plots
cmap = plt.get_cmap('twilight_shifted')
cmap_nums = [tuple(cmap(x)[:3]) for x in np.linspace(0, 1,)]
# Setting options for plots
draw_opts = dict(
    Fullscreen=True,
    deformation=True,
    colors=cmap_nums,
    radius=0.75,
    center=[0.5, 0.5, 0.5],
    settings={
        "Objects": {"Wireframe":False},
        "camera": {
            "transformations":[
                {"type": "rotateX", "angle": -45}
            ]
        },
    },
)


In [3]:
# Setting up the problem we will solve on every level

# Boundary conditions
DIRICHLET = "left|right"

# LHS bilinear form
def poisson_bilinear(a, u, v):
    a += InnerProduct(grad(u), grad(v)) * dx

# RHS linear form
rhs_cf = 0
def poisson_linear(f, u, v):
    f += rhs_cf * v * dx

# Initial iterate
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))


In [4]:

# Call our setup function to put these together
poisson_setup = build_form_setup(bilinear=poisson_bilinear, linear=poisson_linear)

# Define the coarsest mesh for our hierarchy of many sized meshes
N = 16
coarsest_mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
lasagna = build_hierarchy(
    coarsest_mesh,
    poisson_setup,
    n_refines=5,
    order=1,
    dirichlet=DIRICHLET,
    dirichlet_value={"left": 0.0, "right":0.0},
    verbose=True,
)


  lev     ndof              A       P(c->f)    nfree  nfixed
  ---  -------  -------------  ------------  -------  ------
    0      339        339x339             -      305      34  (coarse)
    1     1289      1289x1289      1289x339     1223      66
    2     5025      5025x5025     5025x1289     4895     130
    3    19841    19841x19841    19841x5025    19583     258
    4    78849    78849x78849   78849x19841    78335     514
    5   314369  314369x314369  314369x78849   313343    1026  (fine)


In [5]:
finest = lasagna.finest
finest.set_initial_guess(x0)


In [ ]:

initial_plot = Draw(
        finest.gfu,
        finest.mesh,
        "initial guess (finest)",
        **draw_opts,
        )

In [6]:
vfig = VCycleConfig(
    smoother="native",
    pre_sweeps = 2,
    post_sweeps = 2,
    omega = 1.0,
    coarse_direct = True,
    coarse_sweeps = 20)

garfield = MultigridSolver(lasagna, vfig)
b = lasagna.finest.f.vec
x = lasagna.finest.gfu.vec
layer = lasagna.finest_idx
garfield.v_cycle(layer, b, x, verbose = True)

    [native fwd] sweep   1  ||r_free|| = 3.037190e-01
    [native fwd] sweep   2  ||r_free|| = 2.923672e-01
    [native fwd] sweep   1  ||r_free|| = 5.338579e-01
    [native fwd] sweep   2  ||r_free|| = 5.200262e-01
    [native fwd] sweep   1  ||r_free|| = 9.314909e-01
    [native fwd] sweep   2  ||r_free|| = 8.815671e-01
    [native fwd] sweep   1  ||r_free|| = 1.305401e+00
    [native fwd] sweep   2  ||r_free|| = 1.083495e+00
    [native fwd] sweep   1  ||r_free|| = 8.772159e-01
    [native fwd] sweep   2  ||r_free|| = 6.143149e-01
    [native back] sweep   1  ||r_free|| = 6.307340e-02
    [native back] sweep   2  ||r_free|| = 3.676360e-02
    [native back] sweep   1  ||r_free|| = 2.115635e-01
    [native back] sweep   2  ||r_free|| = 1.511025e-01
    [native back] sweep   1  ||r_free|| = 1.511354e-01
    [native back] sweep   2  ||r_free|| = 1.134000e-01
    [native back] sweep   1  ||r_free|| = 8.542977e-02
    [native back] sweep   2  ||r_free|| = 6.498959e-02
    [native back] sw

In [7]:
post_solve = Draw(
        finest.gfu,
        finest.mesh,
        "our solution approximation",
        **draw_opts,
        )

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…